In [1]:
# =============================================================================
# HUMOB / SIGSPATIAL Cup 2025 - CORRECTED FINAL IMPLEMENTATION
# =============================================================================

import subprocess
import sys
print("Installing requirements...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 
                      "git+https://github.com/yahoojapan/geobleu.git", "tqdm", "scikit-learn", "pandas", "numpy"])

import os
import gc
import time
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

try:
    from geobleu import calc_geobleu_bulk
    print("GeoBLEU imported successfully.\n")
except Exception as e:
    print(f"Failed to import geobleu: {e}")
    calc_geobleu_bulk = None

# =============================================================================
# CONFIGURATION
# =============================================================================
DATA_DIR = "/kaggle/input/humob-data/15313913"
CITIES = ["A"]
COLUMNS = ["uid","d","t","x","y"]
DTYPES = {"uid":"int32","d":"int16","t":"int16","x":"int16","y":"int16"}

TRAIN_DAY_MAX = 60
TEST_DAY_MIN = 61
MASK_VALUE = 999
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

TARGET_RANGES = {
    "A":(147001,150000), "B":(27001,30000), "C":(22001,25000), "D":(17001,20000)
}

# Hyperparameters
TOP_K = 10  # Top K similar days
TRAIN_USER_FRACTION = 0.7  # 70% users for training from days 61-75
POI_TOP_N = 25
PADDING = -999

OUT_DIR = "./results"
os.makedirs(OUT_DIR, exist_ok=True)

MAKE_SUBMISSION = True

# =============================================================================
# UTILITIES
# =============================================================================

def load_city_df(city):
    """Load city data."""
    path = os.path.join(DATA_DIR, f"city_{city}_challengedata.csv")
    if not os.path.exists(path):
        path = os.path.join(DATA_DIR, f"city_{city.lower()}_challengedata.csv")
    print(f"Loading: {path}")
    return pd.read_csv(path, usecols=COLUMNS, dtype=DTYPES)

def to_loc(x, y, max_y):
    """(x,y) -> loc_id"""
    if max_y <= 0:
        max_y = 1
    return int(x * (max_y + 1) + y)

def from_loc(loc, max_y):
    """loc_id -> (x,y)"""
    if max_y <= 0:
        return (0, 0)
    return int(loc // (max_y + 1)), int(loc % (max_y + 1))

# =============================================================================
# PROFILING
# =============================================================================

def build_profiles(df, max_y):
    """Build user profiles efficiently."""
    print(f"Building profiles for {df['uid'].nunique()} users...")
    profiles = {}
    
    for uid, user_df in tqdm(df.groupby('uid'), desc="Profiling"):
        uid = int(uid)
        
        # Day signatures: {day: {time: loc_id}}
        day_sigs = defaultdict(dict)
        loc_freq = Counter()
        hourly_freq = defaultdict(Counter)
        
        for row in user_df.itertuples():
            loc = to_loc(row.x, row.y, max_y)
            day_sigs[row.d][row.t] = loc
            loc_freq[loc] += 1
            hourly_freq[row.t][loc] += 1
        
        # Get fallback location (most frequent overall)
        fallback_loc = loc_freq.most_common(1)[0][0] if loc_freq else to_loc(0, 0, max_y)
        
        profiles[uid] = {
            'days': dict(day_sigs),
            'fallback': fallback_loc,
            'hourly': {t: dict(c) for t, c in hourly_freq.items()},
            'all_locs': set(loc_freq.keys())
        }
        
        del day_sigs, loc_freq, hourly_freq
    
    gc.collect()
    print(f"Profiles built: {len(profiles)}")
    return profiles

def compute_pois(df, max_y, top_n):
    """Compute POIs per timestamp."""
    print(f"Computing top {top_n} POIs per timestamp...")
    pois = {}
    
    for t in range(24):
        t_df = df[df['t'] == t]
        if not t_df.empty:
            locs = [to_loc(row.x, row.y, max_y) for row in t_df.itertuples()]
            counter = Counter(locs)
            pois[t] = {
                'set': set([loc for loc, _ in counter.most_common(top_n)]),
                'top': counter.most_common(1)[0][0] if counter else to_loc(0, 0, max_y)
            }
        else:
            pois[t] = {'set': set(), 'top': to_loc(0, 0, max_y)}
    
    print("POI computation done")
    return pois

# =============================================================================
# SIMILARITY & MATCHING
# =============================================================================

def cosine_similarity_trajectory(clue_dict, day_dict, poi_set=None, use_pois_only=False):
    """
    Calculate cosine similarity between clue trajectory and historical day.
    
    Args:
        clue_dict: {time: loc_id} for current day clue
        day_dict: {time: loc_id} for historical day
        poi_set: Set of POI location IDs (optional)
        use_pois_only: If True, only consider POI matches
    
    Returns:
        similarity score (0-1)
    """
    if not clue_dict or not day_dict:
        return 0.0
    
    # Get common timestamps
    common_times = set(clue_dict.keys()) & set(day_dict.keys())
    if not common_times:
        return 0.0
    
    matches = 0
    total = len(common_times)
    
    for t in common_times:
        clue_loc = clue_dict[t]
        day_loc = day_dict[t]
        
        if use_pois_only and poi_set:
            # Only consider if clue location is a POI
            if clue_loc in poi_set:
                if clue_loc == day_loc:
                    matches += 2  # Exact POI match
                elif day_loc in poi_set:
                    matches += 0.5  # Both are POIs (partial credit)
        else:
            # Standard matching
            if clue_loc == day_loc:
                matches += 1
            elif poi_set and clue_loc in poi_set and day_loc in poi_set:
                matches += 0.3  # Both are POIs (some similarity)
    
    # Normalize by total possible matches
    return matches / (total * 2.0) if use_pois_only else matches / total

def get_top_k_days(profile, clue, max_y, pois=None, use_pois=False, k=TOP_K):
    """
    Get top K most similar historical days.
    
    Returns:
        List of (day_signature, similarity_score) tuples
    """
    if not clue:
        # No clue: return empty list, will use fallbacks
        return []
    
    scores = []
    
    # Determine POI set for current clue timestamps
    poi_set = None
    if use_pois and pois:
        poi_set = set()
        for t in clue.keys():
            if t in pois:
                poi_set.update(pois[t]['set'])
    
    # Score all historical days
    for day, day_sig in profile['days'].items():
        score = cosine_similarity_trajectory(clue, day_sig, poi_set, use_pois)
        if score > 0:
            scores.append((day_sig, score))
    
    # If no matches found, try without POI restriction
    if not scores and use_pois:
        for day, day_sig in profile['days'].items():
            score = cosine_similarity_trajectory(clue, day_sig, None, False)
            if score > 0:
                scores.append((day_sig, score))
    
    # Sort by score and return top K
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:k]

def ensemble_vote(time_slot, candidate_days, clue, profile, max_y, pois):
    """
    Ensemble voting for a single time slot with tie-breaking.
    
    Tie-breaker logic:
    1. If tied, pick location that matches previous timestamp
    2. If still tied, pick from hourly frequency
    3. If still tied, pick most common overall (fallback)
    """
    # Collect votes from candidate days
    votes = []
    for day_sig, score in candidate_days:
        if time_slot in day_sig:
            votes.append(day_sig[time_slot])
    
    if not votes:
        # No votes: use hourly fallback
        if time_slot in profile['hourly'] and profile['hourly'][time_slot]:
            return max(profile['hourly'][time_slot], key=profile['hourly'][time_slot].get)
        else:
            return profile['fallback']
    
    # Count votes
    vote_counts = Counter(votes)
    most_common = vote_counts.most_common()
    top_count = most_common[0][1]
    
    # Get all locations with top vote count (handles ties)
    tied_locs = [loc for loc, count in most_common if count == top_count]
    
    if len(tied_locs) == 1:
        return tied_locs[0]
    
    # TIE-BREAKER 1: Match with previous timestamp in clue
    prev_time = time_slot - 1
    if prev_time in clue:
        prev_loc = clue[prev_time]
        # Check if any candidate day has same previous location
        for day_sig, _ in candidate_days:
            if day_sig.get(prev_time) == prev_loc:
                candidate_loc = day_sig.get(time_slot)
                if candidate_loc in tied_locs:
                    return candidate_loc
    
    # TIE-BREAKER 2: Use hourly frequency
    if time_slot in profile['hourly']:
        hourly = profile['hourly'][time_slot]
        for loc in tied_locs:
            if loc in hourly:
                return loc
    
    # TIE-BREAKER 3: Pick most common overall
    for loc in tied_locs:
        if loc == profile['fallback']:
            return loc
    
    # Final fallback: first in list
    return tied_locs[0]

# =============================================================================
# STRATEGIES
# =============================================================================

def strategy_cosine_basic(uid, clue, slots, profiles, max_y, pois=None, **kw):
    """
    Strategy 1: Basic cosine similarity without POI filtering.
    """
    profile = profiles.get(uid)
    if not profile:
        # No profile: return fallback for all slots
        fallback = to_loc(0, 0, max_y)
        return [(t, fallback) for t in slots]
    
    # Get top K similar days
    top_days = get_top_k_days(profile, clue, max_y, pois=None, use_pois=False)
    
    if not top_days:
        # No similar days: use fallback strategy
        preds = []
        for t in slots:
            if t in profile['hourly'] and profile['hourly'][t]:
                loc = max(profile['hourly'][t], key=profile['hourly'][t].get)
            else:
                loc = profile['fallback']
            preds.append((t, loc))
        return preds
    
    # Ensemble predictions
    preds = []
    for t in slots:
        loc = ensemble_vote(t, top_days, clue, profile, max_y, pois)
        preds.append((t, loc))
        # Update clue with prediction for next timestamp
        clue[t] = loc
    
    return preds

def strategy_cosine_poi(uid, clue, slots, profiles, max_y, pois=None, **kw):
    """
    Strategy 2: Cosine similarity with POI emphasis.
    """
    profile = profiles.get(uid)
    if not profile:
        fallback = to_loc(0, 0, max_y)
        return [(t, fallback) for t in slots]
    
    # Check if clue has any POIs
    has_poi = False
    if pois:
        for t, loc in clue.items():
            if t in pois and loc in pois[t]['set']:
                has_poi = True
                break
    
    # Get top K similar days (with POI emphasis if available)
    top_days = get_top_k_days(profile, clue, max_y, pois=pois, use_pois=has_poi)
    
    if not top_days:
        # No matches: try finding closest POI matches
        if pois and has_poi:
            # Use POI-based fallback
            preds = []
            for t in slots:
                if t in pois:
                    # Use top POI for this timestamp
                    loc = pois[t]['top']
                elif t in profile['hourly'] and profile['hourly'][t]:
                    loc = max(profile['hourly'][t], key=profile['hourly'][t].get)
                else:
                    loc = profile['fallback']
                preds.append((t, loc))
            return preds
        else:
            # Standard fallback
            preds = []
            for t in slots:
                if t in profile['hourly'] and profile['hourly'][t]:
                    loc = max(profile['hourly'][t], key=profile['hourly'][t].get)
                else:
                    loc = profile['fallback']
                preds.append((t, loc))
            return preds
    
    # Ensemble predictions
    preds = []
    for t in slots:
        loc = ensemble_vote(t, top_days, clue, profile, max_y, pois)
        preds.append((t, loc))
        clue[t] = loc
    
    return preds

def strategy_hybrid(uid, clue, slots, profiles, max_y, pois=None, **kw):
    """
    Strategy 3: Hybrid approach - tries POI first, falls back to basic.
    """
    profile = profiles.get(uid)
    if not profile:
        fallback = to_loc(0, 0, max_y)
        return [(t, fallback) for t in slots]
    
    # Try POI-based matching first
    top_days_poi = get_top_k_days(profile, clue, max_y, pois=pois, use_pois=True, k=TOP_K//2)
    
    # Try basic matching
    top_days_basic = get_top_k_days(profile, clue, max_y, pois=None, use_pois=False, k=TOP_K//2)
    
    # Combine candidates (POI matches have higher scores already)
    all_days = top_days_poi + top_days_basic
    
    if not all_days:
        # Complete fallback
        preds = []
        for t in slots:
            if t in profile['hourly'] and profile['hourly'][t]:
                loc = max(profile['hourly'][t], key=profile['hourly'][t].get)
            else:
                loc = profile['fallback']
            preds.append((t, loc))
        return preds
    
    # Remove duplicates (keep highest score)
    seen_days = {}
    for day_sig, score in all_days:
        day_id = id(day_sig)  # Use object id as unique identifier
        if day_id not in seen_days or seen_days[day_id][1] < score:
            seen_days[day_id] = (day_sig, score)
    
    unique_days = list(seen_days.values())
    unique_days.sort(key=lambda x: x[1], reverse=True)
    unique_days = unique_days[:TOP_K]
    
    # Ensemble predictions
    preds = []
    for t in slots:
        loc = ensemble_vote(t, unique_days, clue, profile, max_y, pois)
        preds.append((t, loc))
        clue[t] = loc
    
    return preds

# =============================================================================
# VALIDATION
# =============================================================================

def validate_strategy(val_df, profiles, max_y, pois, strategy_func):
    """
    Proper validation with NO data leakage.
    
    For each (user, day) in validation:
    - Use only revealed timestamps (x != MASK_VALUE) as clue
    - Predict masked timestamps (x == MASK_VALUE)
    - Compare predictions to ground truth
    """
    preds = []
    gts = []
    
    for (uid, day), group in tqdm(val_df.groupby(['uid', 'd']), desc=f"Val {strategy_func.__name__}"):
        uid = int(uid)
        
        # Separate clues from targets
        clue = {}
        targets = {}
        
        for row in group.itertuples():
            if row.x != MASK_VALUE:
                # Revealed timestamp: use as clue
                clue[row.t] = to_loc(row.x, row.y, max_y)
            else:
                # Masked timestamp: this is what we predict
                # In validation, we need ground truth - but it's masked!
                # So we'll use a different approach...
                pass
        
        # For proper validation, we need to artificially mask some data
        # that has ground truth available
        all_known = {row.t: (row.x, row.y) 
                     for row in group.itertuples() 
                     if row.x != MASK_VALUE}
        
        if len(all_known) < 4:  # Need at least some data
            continue
        
        # Split: use first 60% as clue, predict last 40%
        sorted_times = sorted(all_known.keys())
        split_idx = int(len(sorted_times) * 0.6)
        
        clue_times = sorted_times[:split_idx]
        test_times = sorted_times[split_idx:]
        
        if not test_times:
            continue
        
        # Build clue from clue times
        clue = {t: to_loc(all_known[t][0], all_known[t][1], max_y) 
                for t in clue_times}
        
        # Make predictions
        try:
            pred_locs = strategy_func(uid, clue.copy(), test_times, profiles, max_y, pois=pois)
            
            # Collect predictions and ground truth
            for t, pred_loc in pred_locs:
                if t in all_known:
                    pred_x, pred_y = from_loc(pred_loc, max_y)
                    gt_x, gt_y = all_known[t]
                    preds.append((uid, day, t, pred_x, pred_y))
                    gts.append((uid, day, t, gt_x, gt_y))
        except Exception as e:
            # Skip on error
            continue
    
    return preds, gts

def ensemble(pred_map, gts):
    """Weighted ensemble of multiple strategies."""
    if not gts or not pred_map:
        return 0.0
    
    # Weights tuned based on expected performance
    weights = {
        'strategy_cosine_basic': 2.0,
        'strategy_cosine_poi': 2.5,
        'strategy_hybrid': 2.0
    }
    
    votes = defaultdict(Counter)
    gt_map = {(u, d, t): (x, y) for u, d, t, x, y in gts}
    
    for name, preds in pred_map.items():
        w = weights.get(name, 1.0)
        for u, d, t, x, y in preds:
            key = (u, d, t)
            if key in gt_map:
                votes[key][(x, y)] += w
    
    final_preds = []
    final_gts = []
    
    for key, counter in votes.items():
        if counter:
            (x, y), _ = counter.most_common(1)[0]
            final_preds.append((key[0], key[1], key[2], x, y))
            final_gts.append((key[0], key[1], key[2], gt_map[key][0], gt_map[key][1]))
    
    if not final_preds:
        return 0.0
    
    return calc_geobleu_bulk(final_preds, final_gts, processes=1)

# =============================================================================
# SUBMISSION
# =============================================================================

def generate_submission(test_df, profiles, max_y, pois, city):
    """Generate submission with ensemble."""
    print("\nGenerating submission...")
    
    strategies = [
        ('basic', strategy_cosine_basic),
        ('poi', strategy_cosine_poi),
        ('hybrid', strategy_hybrid)
    ]
    
    all_preds = {name: [] for name, _ in strategies}
    
    for name, strategy in strategies:
        print(f"Running {name}...")
        for (uid, day), group in tqdm(test_df.groupby(['uid', 'd']), desc=name):
            uid = int(uid)
            
            clue = {}
            masked = []
            
            for row in group.itertuples():
                if row.x != MASK_VALUE:
                    clue[row.t] = to_loc(row.x, row.y, max_y)
                else:
                    masked.append(row.t)
            
            if masked:
                try:
                    preds = strategy(uid, clue.copy(), masked, profiles, max_y, pois=pois)
                    for t, loc in preds:
                        x, y = from_loc(loc, max_y)
                        all_preds[name].append((uid, day, t, x, y))
                except Exception as e:
                    # Skip on error, use fallback
                    continue
    
    # Ensemble
    print("Ensembling...")
    weights = {'basic': 2.0, 'poi': 2.5, 'hybrid': 2.0}
    votes = defaultdict(Counter)
    
    for name, preds in all_preds.items():
        w = weights[name]
        for u, d, t, x, y in preds:
            votes[(u, d, t)][(x, y)] += w
    
    rows = []
    for (uid, d, t), counter in votes.items():
        if counter:
            (x, y), _ = counter.most_common(1)[0]
            rows.append({'uid': uid, 'd': d, 't': t, 'x': x, 'y': y})
    
    sub_df = pd.DataFrame(rows)
    
    # Filter to target users
    lo, hi = TARGET_RANGES[city]
    sub_df = sub_df[sub_df['uid'].between(lo, hi)]
    
    out_file = os.path.join(OUT_DIR, f"submission_{city}.csv")
    sub_df.to_csv(out_file, index=False)
    print(f"Saved: {out_file} ({len(sub_df)} predictions)")

# =============================================================================
# MAIN
# =============================================================================

def main():
    start = time.time()
    
    for city in CITIES:
        print("\n" + "="*70)
        print(f"CITY {city}")
        print("="*70)
        
        # Load data
        df = load_city_df(city)
        labeled = df[df['x'] != MASK_VALUE]
        
        if labeled.empty:
            print("No labeled data!")
            continue
        
        MAX_Y = int(labeled['y'].max())
        print(f"Grid: {int(labeled['x'].max())} x {MAX_Y}")
        
        # CORRECT SPLIT:
        # Train: days 1-60 (all users) + 70% of users from days 61-75
        # Val:   30% of users from days 61-75
        
        print(f"\nPerforming user-based train/val split...")
        future_labeled = df[(df['d'] >= TEST_DAY_MIN) & (df['x'] != MASK_VALUE)]
        future_users = future_labeled['uid'].unique()
        
        if len(future_users) == 0:
            print("No validation users!")
            continue
        
        # Split users for days 61-75
        train_future_users, val_users = train_test_split(
            future_users,
            train_size=TRAIN_USER_FRACTION,
            random_state=RANDOM_SEED
        )
        
        print(f"Future train users: {len(train_future_users)}, Val users: {len(val_users)}")
        
        # Build training data:
        # - All of days 1-60
        # - train_future_users from days 61-75
        train_data = pd.concat([
            df[(df['d'] <= TRAIN_DAY_MAX) & (df['x'] != MASK_VALUE)],
            future_labeled[future_labeled['uid'].isin(train_future_users)]
        ])
        
        print(f"Training data: {len(train_data)} records")
        
        # Build profiles and POIs
        profiles = build_profiles(train_data, MAX_Y)
        pois = compute_pois(train_data, MAX_Y, POI_TOP_N)
        
        del train_data
        gc.collect()
        
        # Validation data: val_users from days 61-75
        val_data = future_labeled[future_labeled['uid'].isin(val_users)]
        print(f"Validation data: {len(val_data)} records")
        
        if val_data.empty or calc_geobleu_bulk is None:
            print("Cannot run validation!")
            del labeled, future_labeled, val_data
            gc.collect()
        else:
            print("\n" + "-"*70)
            print("VALIDATION (held-out users from days 61-75)")
            print("-"*70)
            
            p1, g1 = validate_strategy(val_data, profiles, MAX_Y, pois, strategy_cosine_basic)
            s1 = calc_geobleu_bulk(p1, g1, processes=1) if p1 else 0.0
            print(f"Strategy 1 (Cosine Basic):    {s1:.6f}")
            
            p2, g2 = validate_strategy(val_data, profiles, MAX_Y, pois, strategy_cosine_poi)
            s2 = calc_geobleu_bulk(p2, g2, processes=1) if p2 else 0.0
            print(f"Strategy 2 (Cosine POI):       {s2:.6f}")
            
            p3, g3 = validate_strategy(val_data, profiles, MAX_Y, pois, strategy_hybrid)
            s3 = calc_geobleu_bulk(p3, g3, processes=1) if p3 else 0.0
            print(f"Strategy 3 (Hybrid):           {s3:.6f}")
            
            pred_map = {
                'strategy_cosine_basic': p1,
                'strategy_cosine_poi': p2,
                'strategy_hybrid': p3
            }
            
            ens = ensemble(pred_map, g1)
            print("-"*70)
            print(f"ENSEMBLE:                      {ens:.6f}")
            print("="*70)
            
            del val_data, p1, p2, p3, g1, g2, g3, pred_map
            gc.collect()
        
        # Generate submission
        if MAKE_SUBMISSION:
            test_data = df[df['d'] >= TEST_DAY_MIN].copy()
            generate_submission(test_data, profiles, MAX_Y, pois, city)
            del test_data
            gc.collect()
        
        del df, labeled, future_labeled, profiles, pois
        gc.collect()
    
    print(f"\nTotal time: {int(time.time() - start)}s")
    print("Done!")

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"\nERROR: {e}")
        import traceback
        traceback.print_exc()

Installing requirements...
GeoBLEU imported successfully.


CITY A
Loading: /kaggle/input/humob-data/15313913/city_A_challengedata.csv
Grid: 200 x 200

Performing user-based train/val split...
Future train users: 102900, Val users: 44100
Training data: 81058680 records
Building profiles for 150000 users...


Profiling:   0%|          | 0/150000 [00:00<?, ?it/s]

Profiles built: 150000
Computing top 25 POIs per timestamp...
POI computation done
Validation data: 5663347 records

----------------------------------------------------------------------
VALIDATION (held-out users from days 61-75)
----------------------------------------------------------------------


Val strategy_cosine_basic:   0%|          | 0/626123 [00:00<?, ?it/s]

Strategy 1 (Cosine Basic):    0.102656


...

Running poi...


poi:   0%|          | 0/2128284 [00:00<?, ?it/s]

Running hybrid...


hybrid:   0%|          | 0/2128284 [00:00<?, ?it/s]

Ensembling...
Saved: ./results/submission_A.csv (320391 predictions)

Total time: 5825s
Done!
